In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
from pycaret.classification import load_model
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

OUTPUT_ROOT = Path("../../outputs")
RESULT_DIR = OUTPUT_ROOT / "model_evaluation" / "oof_predictions"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
RANDOM_STATE = 123

def positive_probability(model, predictors):
    probability = np.asarray(model.predict_proba(predictors))
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "steps"):
        classes = getattr(model.steps[-1][1], "classes_", None)
    classes = list(classes) if classes is not None else [0, 1]
    positive_index = classes.index(1) if 1 in classes else classes.index("1")
    return probability[:, positive_index].astype(float)

manifest = pd.read_csv(OUTPUT_ROOT / "model_evaluation" / "oof_model_manifest.csv")
records = []
for item in manifest.itertuples(index=False):
    predictors = pd.read_csv(Path(item.development_features))
    outcome_frame = pd.read_csv(Path(item.development_outcome))
    target = outcome_frame[item.target_column].astype(int).to_numpy()
    if len(predictors) != len(target):
        raise ValueError(f"Predictor/outcome row mismatch for {item.model_set}/{item.cohort}/{item.outcome}")
    template = load_model(str(Path(item.model_path).with_suffix("")), verbose=False)
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_probability = np.full(len(target), np.nan, dtype=float)
    fold_assignment = np.zeros(len(target), dtype=int)
    for fold, (training_index, held_out_index) in enumerate(splitter.split(predictors, target), 1):
        estimator = clone(template)
        estimator.fit(predictors.iloc[training_index], target[training_index])
        oof_probability[held_out_index] = positive_probability(estimator, predictors.iloc[held_out_index])
        fold_assignment[held_out_index] = fold
        del estimator
        gc.collect()
    if not np.isfinite(oof_probability).all():
        raise ValueError(f"Incomplete OOF predictions for {item.model_set}/{item.cohort}/{item.outcome}")
    destination = RESULT_DIR / f"{item.model_set}_{item.cohort}_{item.outcome}_development_oof_predictions.csv"
    pd.DataFrame({"y_true": target, "y_prob": oof_probability, "fold": fold_assignment}).to_csv(destination, index=False)
    records.append({"model_set": item.model_set, "cohort": item.cohort, "outcome": item.outcome, "development_oof_predictions": str(destination)})

pd.DataFrame(records).to_csv(RESULT_DIR / "oof_prediction_files.csv", index=False)
